In [1]:
import speech_recognition as sr
import os
from pydub import AudioSegment
from pydub.silence import split_on_silence
import time

class SpeechRecognizer:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        self.setup_microphone()
    
    def setup_microphone(self):
        """Adjust for ambient noise and set up microphone."""
        print("Setting up microphone...")
        with self.microphone as source:
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
        print("Microphone setup complete.")
    
    def recognize_from_microphone(self, timeout=5, phrase_time_limit=10):
        """
        Capture speech from the microphone and convert it to text.
        
        Args:
            timeout (int): Seconds to wait for speech before timing out.
            phrase_time_limit (int): Maximum seconds for a phrase.
            
        Returns:
            str: Recognized text or None if recognition fails.
        """
        print(f"Listening... (Timeout in {timeout}s, phrase limit {phrase_time_limit}s)")
        with self.microphone as source:
            try:
                audio = self.recognizer.listen(
                    source, 
                    timeout=timeout, 
                    phrase_time_limit=phrase_time_limit
                )
                text = self.recognizer.recognize_google(audio)
                print(f"Recognized: {text}")
                return text
            except sr.WaitTimeoutError:
                print("Listening timed out while waiting for phrase to start")
                return None
            except sr.UnknownValueError:
                print("Google Speech Recognition could not understand audio")
                return None
            except sr.RequestError as e:
                print(f"Could not request results from Google Speech Recognition service; {e}")
                return None
    
    def recognize_from_file(self, audio_file_path):
        """
        Convert speech from an audio file to text.
        
        Args:
            audio_file_path (str): Path to the audio file.
            
        Returns:
            str: Recognized text or None if recognition fails.
        """
        # Check if file exists
        if not os.path.exists(audio_file_path):
            print(f"File not found: {audio_file_path}")
            return None
        
        # Handle different file formats
        if audio_file_path.endswith('.wav'):
            with sr.AudioFile(audio_file_path) as source:
                audio = self.recognizer.record(source)
        else:
            # Convert non-WAV files to WAV using pydub
            try:
                sound = AudioSegment.from_file(audio_file_path)
                sound.export("temp.wav", format="wav")
                with sr.AudioFile("temp.wav") as source:
                    audio = self.recognizer.record(source)
                os.remove("temp.wav")
            except Exception as e:
                print(f"Error processing audio file: {e}")
                return None
        
        # Perform speech recognition
        try:
            text = self.recognizer.recognize_google(audio)
            print(f"Recognized from file: {text}")
            return text
        except sr.UnknownValueError:
            print("Google Speech Recognition could not understand audio")
            return None
        except sr.RequestError as e:
            print(f"Could not request results from Google Speech Recognition service; {e}")
            return None
    
    def recognize_large_audio(self, audio_file_path, min_silence_len=500, silence_thresh=-40):
        """
        Split large audio file into chunks on silence and recognize each chunk.
        
        Args:
            audio_file_path (str): Path to the audio file.
            min_silence_len (int): Minimum silence length in ms to split on.
            silence_thresh (int): Silence threshold in dBFS.
            
        Returns:
            list: List of recognized text chunks.
        """
        if not os.path.exists(audio_file_path):
            print(f"File not found: {audio_file_path}")
            return []
        
        try:
            # Load audio file
            sound = AudioSegment.from_file(audio_file_path)
            
            # Split audio where silence is longer than min_silence_len
            chunks = split_on_silence(
                sound,
                min_silence_len=min_silence_len,
                silence_thresh=silence_thresh
            )
            
            recognized_texts = []
            
            # Process each chunk
            for i, chunk in enumerate(chunks):
                # Export chunk to WAV
                chunk.export(f"chunk{i}.wav", format="wav")
                
                # Recognize the chunk
                with sr.AudioFile(f"chunk{i}.wav") as source:
                    audio = self.recognizer.record(source)
                    try:
                        text = self.recognizer.recognize_google(audio)
                        recognized_texts.append(text)
                        print(f"Chunk {i}: {text}")
                    except sr.UnknownValueError:
                        print(f"Chunk {i}: [unintelligible]")
                    except sr.RequestError as e:
                        print(f"Chunk {i}: API error - {e}")
                
                # Clean up
                os.remove(f"chunk{i}.wav")
                
            return recognized_texts
        except Exception as e:
            print(f"Error processing large audio file: {e}")
            return []

def main():
    recognizer = SpeechRecognizer()
    
    while True:
        print("\nSpeech Recognition Menu:")
        print("1. Recognize from microphone")
        print("2. Recognize from audio file")
        print("3. Recognize large audio file (chunked)")
        print("4. Exit")
        
        choice = input("Enter your choice (1-4): ")
        
        if choice == "1":
            # Microphone recognition
            print("\nSpeak now...")
            text = recognizer.recognize_from_microphone()
            if text:
                print("\nFinal Result:", text)
        
        elif choice == "2":
            # File recognition
            file_path = input("Enter audio file path: ").strip()
            if file_path:
                text = recognizer.recognize_from_file(file_path)
                if text:
                    print("\nFinal Result:", text)
        
        elif choice == "3":
            # Large file recognition
            file_path = input("Enter large audio file path: ").strip()
            if file_path:
                texts = recognizer.recognize_large_audio(file_path)
                if texts:
                    print("\nFinal Results:")
                    for i, text in enumerate(texts):
                        print(f"Chunk {i}: {text}")
        
        elif choice == "4":
            print("Exiting...")
            break
        
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()

F:\anacondaa\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Setting up microphone...
Microphone setup complete.

Speech Recognition Menu:
1. Recognize from microphone
2. Recognize from audio file
3. Recognize large audio file (chunked)
4. Exit


Enter your choice (1-4):  1



Speak now...
Listening... (Timeout in 5s, phrase limit 10s)
Google Speech Recognition could not understand audio

Speech Recognition Menu:
1. Recognize from microphone
2. Recognize from audio file
3. Recognize large audio file (chunked)
4. Exit


Enter your choice (1-4):  1


...


Speak now...
Listening... (Timeout in 5s, phrase limit 10s)
Recognized: hello hello hay hello

Final Result: hello hello hay hello

Speech Recognition Menu:
1. Recognize from microphone
2. Recognize from audio file
3. Recognize large audio file (chunked)
4. Exit


Enter your choice (1-4):  exit


Invalid choice. Please try again.

Speech Recognition Menu:
1. Recognize from microphone
2. Recognize from audio file
3. Recognize large audio file (chunked)
4. Exit


Enter your choice (1-4):  4


Exiting...


In [3]:
import speech_recognition as sr
import webbrowser
import time
import os
from datetime import datetime
import pyttsx3

class VoiceAssistant:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        self.engine = pyttsx3.init()
        self.setup_voice_engine()
        self.setup_microphone()
        
        # Command mappings
        self.commands = {
            "open google": self.open_google,
            "search google": self.search_google,
            "open youtube": self.open_youtube,
            "search youtube": self.search_youtube,
            "what time is it": self.tell_time,
            "exit": self.exit_program
        }
    
    def setup_voice_engine(self):
        """Set up the text-to-speech engine"""
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[0].id)  # 0 for male, 1 for female
        self.engine.setProperty('rate', 150)  # Speed of speech
    
    def setup_microphone(self):
        """Adjust for ambient noise and set up microphone."""
        print("Setting up microphone...")
        with self.microphone as source:
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
        print("Microphone setup complete.")
    
    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
    
    def listen(self, timeout=5, phrase_time_limit=8):
        """
        Listen to microphone input and return recognized text.
        
        Args:
            timeout (int): Seconds to wait for speech before timing out.
            phrase_time_limit (int): Maximum seconds for a phrase.
            
        Returns:
            str: Recognized text or None if recognition fails.
        """
        with self.microphone as source:
            print("Listening...")
            try:
                audio = self.recognizer.listen(
                    source, 
                    timeout=timeout, 
                    phrase_time_limit=phrase_time_limit
                )
                text = self.recognizer.recognize_google(audio).lower()
                print(f"You said: {text}")
                return text
            except sr.WaitTimeoutError:
                print("Listening timed out")
                return None
            except sr.UnknownValueError:
                print("Could not understand audio")
                return None
            except sr.RequestError as e:
                print(f"Could not request results; {e}")
                return None
    
    def open_google(self):
        """Open Google in the default web browser"""
        self.speak("Opening Google")
        webbrowser.open("https://www.google.com")
    
    def search_google(self):
        """Search Google based on voice input"""
        self.speak("What would you like to search on Google?")
        query = self.listen()
        if query:
            url = f"https://www.google.com/search?q={query.replace(' ', '+')}"
            self.speak(f"Searching Google for {query}")
            webbrowser.open(url)
    
    def open_youtube(self):
        """Open YouTube in the default web browser"""
        self.speak("Opening YouTube")
        webbrowser.open("https://www.youtube.com")
    
    def search_youtube(self):
        """Search YouTube based on voice input"""
        self.speak("What would you like to search on YouTube?")
        query = self.listen()
        if query:
            url = f"https://www.youtube.com/results?search_query={query.replace(' ', '+')}"
            self.speak(f"Searching YouTube for {query}")
            webbrowser.open(url)
    
    def tell_time(self):
        """Speak the current time"""
        current_time = datetime.now().strftime("%I:%M %p")
        self.speak(f"The current time is {current_time}")
    
    def exit_program(self):
        """Exit the program"""
        self.speak("Goodbye!")
        exit()
    
    def process_command(self, command):
        """Process the recognized command"""
        # Check for exact matches first
        if command in self.commands:
            self.commands[command]()
            return True
        
        # Check for partial matches
        for cmd in self.commands:
            if cmd in command:
                self.commands[cmd]()
                return True
        
        return False
    
    def run(self):
        """Main execution loop"""
        self.speak("Hello! I'm your voice assistant. How can I help you?")
        
        while True:
            command = self.listen()
            
            if command:
                if not self.process_command(command):
                    self.speak("I didn't understand that command. Please try again.")
                    print("Available commands:")
                    for cmd in self.commands:
                        print(f"- {cmd}")

if __name__ == "__main__":
    assistant = VoiceAssistant()
    try:
        assistant.run()
    except KeyboardInterrupt:
        assistant.speak("Goodbye!")
        print("\nProgram terminated by user")

Setting up microphone...
Microphone setup complete.
Assistant: Hello! I'm your voice assistant. How can I help you?
Listening...
Could not understand audio
Listening...
You said: what ise your name
Assistant: I didn't understand that command. Please try again.
Available commands:
- open google
- search google
- open youtube
- search youtube
- what time is it
- exit
Listening...
Could not understand audio
Listening...
You said: what ise your name what is your name
Assistant: I didn't understand that command. Please try again.
Available commands:
- open google
- search google
- open youtube
- search youtube
- what time is it
- exit
Listening...
You said: open google
Assistant: Opening Google
Listening...
Could not understand audio
Listening...
Could not understand audio
Listening...
Could not understand audio
Listening...
Listening timed out
Listening...
You said: open youtube
Assistant: Opening YouTube
Listening...
Could not understand audio
Listening...
Could not understand audio
Liste

ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

In [ ]:
import speech_recognition as sr
import webbrowser
import datetime
import pyttsx3
import time

class VoiceAssistant:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        self.engine = pyttsx3.init()
        
        # Configure voice properties
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[0].id)  # 0 for male, 1 for female
        self.engine.setProperty('rate', 150)  # Speech speed
        
        # Define commands
        self.commands = {
            "open google": self.open_google,
            "open youtube": self.open_youtube,
            "open facebook": self.open_facebook,
            "open twitter": self.open_twitter,
            "open instagram": self.open_instagram,
            "open gmail": self.open_gmail,
            "what time is it": self.tell_time,
            "what is today's date": self.tell_date,
            "what day is today": self.tell_day,
            "exit": self.exit_program
        }
    
    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
    
    def listen(self):
        """Listen to microphone input and return recognized text"""
        with self.microphone as source:
            print("Listening...")
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
            try:
                audio = self.recognizer.listen(source, timeout=5, phrase_time_limit=8)
                text = self.recognizer.recognize_google(audio).lower()
                print(f"You said: {text}")
                return text
            except sr.WaitTimeoutError:
                self.speak("Sorry, I didn't hear anything.")
                return None
            except sr.UnknownValueError:
                self.speak("Sorry, I didn't understand that.")
                return None
            except sr.RequestError:
                self.speak("Sorry, my speech service is down.")
                return None
    
    def open_google(self):
        self.speak("Opening Google")
        webbrowser.open("https://www.google.com")
    
    def open_youtube(self):
        self.speak("Opening YouTube")
        webbrowser.open("https://www.youtube.com")
    
    def open_facebook(self):
        self.speak("Opening Facebook")
        webbrowser.open("https://www.facebook.com")
    
    def open_twitter(self):
        self.speak("Opening Twitter")
        webbrowser.open("https://www.twitter.com")
    
    def open_instagram(self):
        self.speak("Opening Instagram")
        webbrowser.open("https://www.instagram.com")
    
    def open_gmail(self):
        self.speak("Opening Gmail")
        webbrowser.open("https://mail.google.com")
    
    def tell_time(self):
        current_time = datetime.datetime.now().strftime("%I:%M %p")
        self.speak(f"The current time is {current_time}")
    
    def tell_date(self):
        current_date = datetime.datetime.now().strftime("%B %d, %Y")
        self.speak(f"Today's date is {current_date}")
    
    def tell_day(self):
        current_day = datetime.datetime.now().strftime("%A")
        self.speak(f"Today is {current_day}")
    
    def exit_program(self):
        self.speak("Goodbye! Have a great day.")
        exit()
    
    def run(self):
        self.speak("Hello! I am your voice assistant. How can I help you?")
        
        while True:
            command = self.listen()
            
            if command:
                if command in self.commands:
                    self.commands[command]()
                else:
                    self.speak("Sorry, I don't know that command. Try again.")
                    print("Available commands:")
                    for cmd in self.commands:
                        print(f"- {cmd}")

if __name__ == "__main__":
    assistant = VoiceAssistant()
    try:
        assistant.run()
    except KeyboardInterrupt:
        assistant.speak("Goodbye!")
        print("\nAssistant terminated.")

Assistant: Hello! I am your voice assistant. How can I help you?
Listening...
You said: what ise your name
Assistant: Sorry, I don't know that command. Try again.
Available commands:
- open google
- open youtube
- open facebook
- open twitter
- open instagram
- open gmail
- what time is it
- what is today's date
- what day is today
- exit
Listening...
Assistant: Sorry, I didn't understand that.
Listening...
You said: what is time today
Assistant: Sorry, I don't know that command. Try again.
Available commands:
- open google
- open youtube
- open facebook
- open twitter
- open instagram
- open gmail
- what time is it
- what is today's date
- what day is today
- exit
Listening...
You said: today's date
Assistant: Sorry, I don't know that command. Try again.
Available commands:
- open google
- open youtube
- open facebook
- open twitter
- open instagram
- open gmail
- what time is it
- what is today's date
- what day is today
- exit
Listening...
You said: what is today's date
Assistant: T

In [1]:
import speech_recognition as sr
import webbrowser
import datetime
import pyttsx3
import sys
import time

class VoiceAssistant:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        self.engine = pyttsx3.init()
        self.running = True  # Control flag for the main loop
        
        # Configure voice properties
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[0].id)
        self.engine.setProperty('rate', 150)
        
        # Define commands
        self.commands = {
            "open google": self.open_google,
            "open youtube": self.open_youtube,
            "open facebook": self.open_facebook,
            "open twitter": self.open_twitter,
            "open instagram": self.open_instagram,
            "open gmail": self.open_gmail,
            "what time is it": self.tell_time,
            "what is today's date": self.tell_date,
            "what day is today": self.tell_day,
            "exit": self.exit_program,
            "quit": self.exit_program,
            "stop": self.exit_program
        }
    
    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
    
    def listen(self):
        """Listen to microphone input and return recognized text"""
        with self.microphone as source:
            print("Listening... (Say 'exit' to quit)")
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
            try:
                audio = self.recognizer.listen(source, timeout=5, phrase_time_limit=8)
                text = self.recognizer.recognize_google(audio).lower()
                print(f"You said: {text}")
                return text
            except sr.WaitTimeoutError:
                return None
            except sr.UnknownValueError:
                self.speak("Sorry, I didn't understand that.")
                return None
            except sr.RequestError:
                self.speak("Sorry, my speech service is down.")
                return None
    
    # Website opening functions
    def open_google(self): webbrowser.open("https://www.google.com")
    def open_youtube(self): webbrowser.open("https://www.youtube.com")
    def open_facebook(self): webbrowser.open("https://www.facebook.com")
    def open_twitter(self): webbrowser.open("https://www.twitter.com")
    def open_instagram(self): webbrowser.open("https://www.instagram.com")
    def open_gmail(self): webbrowser.open("https://mail.google.com")
    
    # Time/date functions
    def tell_time(self): self.speak(datetime.datetime.now().strftime("The time is %I:%M %p"))
    def tell_date(self): self.speak(datetime.datetime.now().strftime("Today's date is %B %d, %Y"))
    def tell_day(self): self.speak(datetime.datetime.now().strftime("Today is %A"))
    
    def exit_program(self):
        """Clean exit function"""
        self.speak("Goodbye! Have a great day.")
        self.running = False
    
    def run(self):
        """Main execution loop"""
        self.speak("Hello! I am your voice assistant. How can I help you?")
        
        while self.running:
            try:
                command = self.listen()
                
                if command:
                    if command in self.commands:
                        self.commands[command]()
                    else:
                        self.speak("Sorry, I don't know that command.")
            except KeyboardInterrupt:
                self.exit_program()
                break
            except Exception as e:
                print(f"Error: {e}")
                continue

# Create and run the assistant
assistant = VoiceAssistant()
assistant.run()

Assistant: Hello! I am your voice assistant. How can I help you?
Listening... (Say 'exit' to quit)
You said: time is it
Assistant: Sorry, I don't know that command.
Listening... (Say 'exit' to quit)
You said: what time is it
Assistant: The time is 12:39 AM
Listening... (Say 'exit' to quit)
You said: what is date today
Assistant: Sorry, I don't know that command.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Assistant: Sorry, my speech service is down.
Listening... (Say 'exit' to quit)
Listening... (Say 'exit' to quit)
Assistant: Sorry, I didn't understand that.
Listening... (Say 'exit' to quit)
You said: what


In [ ]:
import speech_recognition as sr
import webbrowser
import datetime
import pyttsx3
import time
import sys
from urllib.parse import quote

class VoiceAssistant:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        self.microphone = sr.Microphone()
        self.engine = pyttsx3.init()
        self.running = True
        
        # Voice configuration
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[0].id)  # 0 for male, 1 for female
        self.engine.setProperty('rate', 150)
        
        # Command dictionary
        self.commands = {
            "open google": lambda: self.open_website("https://www.google.com", "Google"),
            "open youtube": lambda: self.open_website("https://www.youtube.com", "YouTube"),
            "open facebook": lambda: self.open_website("https://www.facebook.com", "Facebook"),
            "open twitter": lambda: self.open_website("https://www.twitter.com", "Twitter"),
            "open instagram": lambda: self.open_website("https://www.instagram.com", "Instagram"),
            "open email": lambda: self.open_website("https://mail.google.com", "Gmail"),
            "search google": self.search_google,
            "search youtube": self.search_youtube,
            "search facebook": self.search_facebook,
            "search twitter": self.search_twitter,
            "search instagram": self.search_instagram,
            "what time is it": self.tell_time,
            "what's the date": self.tell_date,
            "what day is it": self.tell_day,
            "exit": self.exit_program,
            "quit": self.exit_program,
            "stop": self.exit_program
        }
    
    def speak(self, text):
        """Convert text to speech"""
        print(f"Assistant: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
    
    def listen(self):
        """Listen to microphone input"""
        with self.microphone as source:
            print("\nListening... (Say 'exit' to quit)")
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
            try:
                audio = self.recognizer.listen(source, timeout=5, phrase_time_limit=8)
                text = self.recognizer.recognize_google(audio).lower()
                print(f"You said: {text}")
                return text
            except sr.WaitTimeoutError:
                return None
            except sr.UnknownValueError:
                self.speak("Sorry, I didn't understand that.")
                return None
            except sr.RequestError:
                self.speak("Sorry, my speech service is down.")
                return None
    
    def open_website(self, url, name):
        """Open a website"""
        self.speak(f"Opening {name}")
        webbrowser.open(url)
    
    def search_google(self):
        """Search on Google"""
        self.speak("What would you like to search on Google?")
        query = self.listen()
        if query:
            url = f"https://www.google.com/search?q={quote(query)}"
            self.open_website(url, f"Google for {query}")
    
    def search_youtube(self):
        """Search on YouTube"""
        self.speak("What would you like to search on YouTube?")
        query = self.listen()
        if query:
            url = f"https://www.youtube.com/results?search_query={quote(query)}"
            self.open_website(url, f"YouTube for {query}")
    
    def search_facebook(self):
        """Search on Facebook"""
        self.speak("What would you like to search on Facebook?")
        query = self.listen()
        if query:
            url = f"https://www.facebook.com/search/top/?q={quote(query)}"
            self.open_website(url, f"Facebook for {query}")
    
    def search_twitter(self):
        """Search on Twitter"""
        self.speak("What would you like to search on Twitter?")
        query = self.listen()
        if query:
            url = f"https://twitter.com/search?q={quote(query)}"
            self.open_website(url, f"Twitter for {query}")
    
    def search_instagram(self):
        """Search on Instagram"""
        self.speak("What would you like to search on Instagram?")
        query = self.listen()
        if query:
            url = f"https://www.instagram.com/explore/tags/{quote(query)}/"
            self.open_website(url, f"Instagram for {query}")
    
    def tell_time(self):
        """Tell current time"""
        current_time = datetime.datetime.now().strftime("%I:%M %p")
        self.speak(f"The time is {current_time}")
    
    def tell_date(self):
        """Tell current date"""
        current_date = datetime.datetime.now().strftime("%B %d, %Y")
        self.speak(f"Today's date is {current_date}")
    
    def tell_day(self):
        """Tell current day"""
        current_day = datetime.datetime.now().strftime("%A")
        self.speak(f"Today is {current_day}")
    
    def exit_program(self):
        """Exit the program"""
        self.speak("Goodbye! Have a great day.")
        self.running = False
    
    def run(self):
        """Main execution loop"""
        self.speak("Hello! I'm your voice assistant. How can I help you today?")
        
        while self.running:
            try:
                command = self.listen()
                
                if command:
                    # Check for exact matches first
                    if command in self.commands:
                        self.commands[command]()
                    else:
                        # Check for partial matches
                        matched = False
                        for cmd in self.commands:
                            if cmd in command:
                                self.commands[cmd]()
                                matched = True
                                break
                        
                        if not matched:
                            self.speak("Sorry, I don't understand that command.")
                            print("\nAvailable commands:")
                            for cmd in sorted(self.commands.keys()):
                                print(f"- {cmd}")
            
            except KeyboardInterrupt:
                self.exit_program()
            except Exception as e:
                print(f"Error: {e}")
                continue

# Create and run the assistant
if __name__ == "__main__":
    try:
        assistant = VoiceAssistant()
        assistant.run()
    except Exception as e:
        print(f"Error starting assistant: {e}")
    finally:
        sys.exit(0)

Assistant: Hello! I'm your voice assistant. How can I help you today?

Listening... (Say 'exit' to quit)
You said: what is day today
Assistant: Sorry, I don't understand that command.

Available commands:
- exit
- open email
- open facebook
- open google
- open instagram
- open twitter
- open youtube
- quit
- search facebook
- search google
- search instagram
- search twitter
- search youtube
- stop
- what day is it
- what time is it
- what's the date

Listening... (Say 'exit' to quit)
You said: day is today
Assistant: Sorry, I don't understand that command.

Available commands:
- exit
- open email
- open facebook
- open google
- open instagram
- open twitter
- open youtube
- quit
- search facebook
- search google
- search instagram
- search twitter
- search youtube
- stop
- what day is it
- what time is it
- what's the date

Listening... (Say 'exit' to quit)

Listening... (Say 'exit' to quit)

Listening... (Say 'exit' to quit)
You said: what day is it
Assistant: Today is Tuesday

List